# Weekly Reading Picks: pipeline walkthrough

This notebook runs the same code as the web app, one step at a time, so you can see exactly how a week's
submissions become the Thursday Word document with the top 5 in bold.

| Step | What happens | App file |
|---|---|---|
| 1 | Clean and de-duplicate the submitted links | `fetcher.py` |
| 2 | Open each link, extract the text, check for a paywall | `fetcher.py` |
| 3 | Score each article against Adam's criteria | `scorer.py` |
| 4 | Rule out ineligible links and rank the top 5 | `scorer.py` |
| 5 | Build the Word document | `report.py` |
| 6 | Preview the Monday reminder and Thursday emails | `jobs.py`, `mailer.py` |
| 7 | Check the weekly deadline logic | `cycle.py` |

It uses its own database in `notebook_data/`, so it never touches the live app's data, and it never sends
email unless you turn that on below.

**How to run:** put this notebook in the project folder (next to `app.py`), fill in the inputs in the next
two cells, then choose *Run All*.

## Inputs
Add real links here. Each row is `(submitter name, link, optional note)`.

In [ ]:
SUBMISSIONS = [
    ("Priya Shah", "https://www.example.com/replace-with-a-real-article", "Great long-term perspective"),
    ("Marcus Lee", "https://www.example.com/another-real-article", ""),
    # ("Name", "https://...", "why it's worth reading"),
]

# Adam's description of the firm's approach. The AI compares every article to this.
WORLDVIEW = """
We are patient, long-term investors focused on fundamentals and valuation.
We are skeptical of short-term forecasts and market noise.
We want clients to understand risk, compounding, and the value of discipline.
"""

# Optional but recommended: paste two or three past Friday client emails
PAST_EXAMPLES = """
"""

USE_SAMPLE_DATA = True   # True = fictional offline sample (no internet or API key needed). False = your links above.
SEND_EMAILS = False      # Keep False to preview emails without sending them

## Setup
Loads the app's modules and settings from `.env`, and points storage at a separate notebook database.

In [ ]:
import os, sys, glob, email
from email import policy
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())
from config import Config

Config.DATABASE_PATH = "notebook_data/notebook.db"
Config.OUTPUT_DIR = "notebook_data/reports"
Config.OUTBOX_DIR = "notebook_data/outbox"
if not SEND_EMAILS:
    Config.SMTP_HOST = ""  # emails are saved to notebook_data/outbox instead of sent

import db, cycle as cyc, fetcher, scorer, report, jobs, manage
db.init_db()

pd.set_option("display.max_colwidth", 80)
print("AI scoring:", "ON (" + Config.ANTHROPIC_MODEL + ")" if Config.ANTHROPIC_API_KEY else "OFF, using basic rules")
print("Email:", "will SEND" if Config.SMTP_HOST else "preview only (saved to notebook_data/outbox)")

Save the settings the app would normally take from its Settings page.

In [ ]:
if USE_SAMPLE_DATA:
    SUBMISSIONS = [(who, url, "Great long-term perspective." if "dividend" in url else "")
                   for who, url, *_ in manage.DEMO]

team = sorted({name for name, _, _ in SUBMISSIONS})
db.save_settings({
    "team_members": "\n".join(f"{n}, {n.split()[0].lower()}@example.com" for n in team),
    "report_recipients": "adam@example.com",
    "worldview": WORLDVIEW.strip(),
    "past_examples": PAST_EXAMPLES.strip(),
})
settings = db.get_settings()
WEEK = cyc.open_cycle()
print(f"Week closing {cyc.label(WEEK)} | {len(SUBMISSIONS)} submissions from {len(team)} people")

## Step 1: Clean the links
Tracking codes like `utm_source` are stripped so the same article sent twice (or by two people) is
recognized as one link. Invalid entries are dropped, just like on the submission page.

In [ ]:
rows = []
for name, raw, note in SUBMISSIONS:
    url = fetcher.normalize_url(raw)
    rows.append({"submitter": name, "submitted": raw, "cleaned": url, "valid": fetcher.is_valid_url(url), "note": note})
    if fetcher.is_valid_url(url):
        db.add_submission(WEEK, name, url, note)

links = pd.DataFrame(rows)
print(f"{links['cleaned'].nunique()} unique links out of {len(links)} submitted")
links

## Step 2: Open one article and look inside
This is what the app sees for each link: the extracted text, length, readability, and the paywall check.
Skipped in sample mode because the sample links are fictional.

In [ ]:
if USE_SAMPLE_DATA:
    print("Sample mode: switch USE_SAMPLE_DATA to False to inspect a real article.")
    extra = []
    print("Paywall check on a known subscription site:", fetcher.known_paywall("https://www.wsj.com/finance/x"))
else:
    extra = [d for d in settings["extra_paywall_domains"].split(",") if d.strip()]
    art = fetcher.fetch_article(links.loc[links["valid"], "cleaned"].iloc[0], extra)
    print("Title:        ", art["title"])
    print("Site:         ", art["site"] or art["domain"])
    print("Words:        ", art["word_count"])
    print("Readability:  ", art["reading_ease"], "(60+ is plain English, under 30 is academic)")
    print("Paywall:      ", art["paywall"], "|", art["paywall_reason"] or "no signs of a paywall")
    print("Error:        ", art["error"] or "none")
    print("\nFirst 600 characters:\n", art["text"][:600])

## Step 3: Score every article
Each article gets a 1 to 5 score on five criteria from Adam's email, combined into a score out of 100:

| Criterion | Weight |
|---|---|
| Fits our view of markets | 30% |
| Fresh angle (not widely covered) | 25% |
| Finance, investing, or economics | 20% |
| Right depth (not too technical or short) | 15% |
| Non-partisan | 10% |

Results are cached, so re-running this cell is instant and doesn't cost anything.

In [ ]:
if USE_SAMPLE_DATA:
    manage.seed_demo_analyses(settings)  # loads pre-written sample scores instead of calling the AI

items = jobs.collect(WEEK, settings)  # fetch + score everything (in parallel), then rank

scores = pd.DataFrame([{
    "title": i["title"][:60],
    "sent by": ", ".join(i["submitters"]),
    **({scorer.CRITERIA_LABELS[k]: i["scores"].get(k) for k in scorer.WEIGHTS} if i["scores"] else {}),
    "score": i["overall"],
    "paywall": i["paywall"],
    "eligible": i["eligible"],
    "method": i["method"],
} for i in items]).sort_values("score", ascending=False, na_position="last")
scores

## Step 4: Hard rules and the top 5
A link is never suggested if it's behind a paywall, strongly partisan, off-topic, too technical, too
short, or unreadable. The remaining links are ranked by score; ties go to links more than one person sent.

In [ ]:
top = sorted([i for i in items if i["rank"]], key=lambda i: i["rank"])
print(f"Suggested top {len(top)}")
display(pd.DataFrame([{"rank": i["rank"], "title": i["title"], "sent by": ", ".join(i["submitters"]),
                       "score": i["overall"], "why": i["why"]} for i in top]).set_index("rank"))

excluded = [i for i in items if not i["eligible"]]
print(f"\nRuled out ({len(excluded)})")
display(pd.DataFrame([{"title": i["title"], "reason": "; ".join(i["exclude_reasons"])} for i in excluded]))

In [ ]:
chart = scores.dropna(subset=["score"]).sort_values("score")
colors = ["#9a6b22" if t in {i["title"][:60] for i in top} else ("#c9c9c9" if not e else "#5b6b6a")
          for t, e in zip(chart["title"], chart["eligible"])]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(chart) + 1))
ax.barh(chart["title"], chart["score"], color=colors)
ax.set_xlabel("Score out of 100")
ax.set_title("Article scores (gold = suggested top 5, light grey = ruled out)")
ax.set_xlim(0, 100)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## Step 5: Build the Word document
The exact file Adam receives on Thursday.

In [ ]:
path, _ = jobs.build_report_file(WEEK)
print("Saved:", os.path.abspath(path))

from docx import Document
doc = Document(path)
bold_links = {r.text for p in doc.paragraphs for r in p.runs if r.bold and r.text.startswith("http")}
print(f"Unique bold links in the document: {len(bold_links)} (matches the top {len(top)}: {bold_links == {i['url'] for i in top}})")

## Step 6: Preview the emails
With `SEND_EMAILS = False`, these are saved to `notebook_data/outbox` instead of sent.

In [ ]:
def show_latest_email():
    latest = max(glob.glob(os.path.join(Config.OUTBOX_DIR, "*.eml")), key=os.path.getmtime)
    with open(latest, "rb") as fh:
        msg = email.message_from_binary_file(fh, policy=policy.default)
    print("To:     ", msg["To"])
    print("Subject:", msg["Subject"])
    print("Files:  ", [a.get_filename() for a in msg.iter_attachments()] or "none")
    print()
    print(msg.get_body(preferencelist=("plain",)).get_content())

print(jobs.send_reminder(WEEK)[1]); print("-" * 70)
show_latest_email()

In [ ]:
print(jobs.send_report(WEEK)[1]); print("-" * 70)
show_latest_email()

## Step 7: The weekly deadline
Every submission belongs to the Thursday-noon deadline it counts toward. The reminder goes out Monday
morning, and the report goes out right after the deadline.

In [ ]:
from datetime import datetime
tz = cyc.tz()
examples = [datetime(2026, 9, 21, 9, 0, tzinfo=tz), datetime(2026, 9, 24, 11, 59, tzinfo=tz),
            datetime(2026, 9, 24, 12, 1, tzinfo=tz), datetime(2026, 9, 26, 15, 0, tzinfo=tz)]
pd.DataFrame([{"submitted at": f"{d:%a %b %d, %I:%M %p}", "counts toward deadline": cyc.label(cyc.open_cycle(d)),
               "reminder due?": cyc.in_reminder_window(d)} for d in examples])

## Start over
Run this to clear the notebook's database and files, for example before switching from sample data to real links.

In [ ]:
# import shutil; shutil.rmtree("notebook_data", ignore_errors=True); print("Cleared notebook_data/")